# Ajuste de Curva pelo Método dos Mínimos Quadrados (MMQ)

**Disciplina**: Cálculo Numérico  
**Professor**: Marcos Maia  
**Autor**: Yann Keven Jordão Leão (Engenharia da Computação - UFRPE)

Este notebook tem como objetivo aplicar o Método dos Mínimos Quadrados (MMQ) para ajustar uma função aos dados históricos de fechamento diário do contrato futuro do índice Bovespa (WIN), entre os dias **08/05/2025** e **06/06/2025**.

A partir desse ajuste, buscamos:
- Analisar o comportamento do índice no período.
- Estimar o valor do fechamento para o dia seguinte (07/06/2025).
- Avaliar a qualidade do ajuste por meio da análise de resíduos.
- Construir um intervalo de confiança para a previsão, considerando uma regressão linear simples.

Todos os dados referem-se a **dias úteis**, portanto há lacunas nos finais de semana e feriados.


## Conjunto de Dados

A tabela a seguir apresenta os valores de fechamento diário do índice WIN (contrato futuro do Ibovespa) no período entre **08/05/2025** e **06/06/2025**. 

Para facilitar a manipulação computacional, os dias foram codificados de forma crescente a partir do mais recente:

- O dia **0** corresponde a **06/06/2025** (último valor da série).
- O dia **–1** representa o dia anterior (05/06/2025), e assim por diante.

Essa escolha facilita a previsão para o dia seguinte, que corresponde a **x = 1**.

| Dias | Data       | WIN (Fechamento) |
|------|------------|------------------|
| 0    | 06/06/2025 | 136.772          |
| –1   | 05/06/2025 | 136.819          |
| –2   | 04/06/2025 | 137.622          |
| –3   | 03/06/2025 | 138.295          |
| –4   | 02/06/2025 | 137.558          |
| –5   | 30/05/2025 | 138.280          |
| –6   | 29/05/2025 | 139.588          |
| –7   | 28/05/2025 | 140.030          |
| –8   | 27/05/2025 | 140.726          |
| –9   | 26/05/2025 | 139.336          |
| –10  | 23/05/2025 | 138.954          |
| –11  | 22/05/2025 | 138.432          |
| –12  | 21/05/2025 | 139.060          |
| –13  | 20/05/2025 | 141.527          |
| –14  | 19/05/2025 | 141.107          |
| –15  | 16/05/2025 | 140.692          |
| –16  | 15/05/2025 | 141.039          |
| –17  | 14/05/2025 | 140.072          |
| –18  | 13/05/2025 | 140.788          |
| –19  | 12/05/2025 | 138.426          |
| –20  | 09/05/2025 | 138.250          |
| –21  | 08/05/2025 | 138.221          |

Estes dados servirão de base para os ajustes e previsões a seguir.


In [ ]:
# Valores da tabela
import numpy as np
dias = np.array(range(0, -22, -1))  
win = np.array([
    136.772, 136.819, 137.622, 138.295, 137.558,
    138.280, 139.588, 140.030, 140.726, 139.336,
    138.954, 138.432, 139.060, 141.527, 141.107,
    140.692, 141.039, 140.072, 140.788, 138.426,
    138.250, 138.221 
])

In [ ]:
# Gráfico de dispersão
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 5))
plt.scatter(dias, win, label='Dados Reais', color='blue')

plt.xticks(dias)
plt.xlabel('Dias (x)')
plt.ylabel('Fechamento WIN')

plt.title('Gráfico de Dispersão - WIN nos últimos 21 dias')
plt.legend()
plt.show()

## O Método dos Mínimos Quadrados (MMQ)

O Método dos Mínimos Quadrados (MMQ) é utilizado para encontrar uma função $\phi(x)$ que se aproxime dos dados observados $(x_i, y_i)$, minimizando o erro entre os valores reais e os valores ajustados.

### Construção da função ajustada

A ideia é escrever $\phi(x)$ como uma **combinação linear** de funções base $g_j(x)$:

$$
\phi(x) = \alpha_1 g_1(x) + \alpha_2 g_2(x) + \dots + \alpha_n g_n(x)
$$

O objetivo é determinar os coeficientes $\alpha_j$ que tornam essa aproximação a melhor possível, no sentido de minimizar o erro quadrático.

### Sistema $A \cdot \alpha = b$

Para isso, montamos um sistema linear da forma:

$$
A \cdot \alpha = b
$$

Onde:

- $A$ é uma matriz em que cada elemento $A_{ij}$ é o somatório do produto das funções base:  
  $$
  A_{ij} = \sum_{k=1}^{n} g_i(x_k) \cdot g_j(x_k)
  $$

- $b$ é um vetor cujos elementos são os produtos das funções base com os valores observados:  
  $$
  b_i = \sum_{k=1}^{n} g_i(x_k) \cdot y_k
  $$

- $\alpha = [\alpha_1, \alpha_2, \dots, \alpha_n]$ é o vetor de coeficientes que queremos encontrar.

Essa formulação é baseada em **produtos internos** entre os vetores das funções base e os dados observados, e garante que os resíduos sejam ortogonais ao espaço gerado pelas funções $g_j(x)$.

### Solução

Ao resolver o sistema $A \cdot \alpha = b$, obtemos os coeficientes que definem $\phi(x)$. Essa função será usada como modelo ajustado para análise e previsão.

In [ ]:
# Método dos mínimos quadrados
def mmq(funcoes, x_data, y_data):
    n = len(funcoes)
    A = np.zeros((n, n))
    b = np.zeros(n)

    for i in range(n):
        for j in range(n):
            A[i, j] = sum(funcoes[i](x) * funcoes[j](x) for x in x_data)
        b[i] = sum(y_data[k] * funcoes[i](x_data[k]) for k in range(len(x_data)))

    alfas = np.linalg.solve(A, b)

    # Determinando phi
    def phi(x):
        return sum(alfas[i] * funcoes[i](x) for i in range(n))

    return phi, alfas

In [ ]:
# Determinado as funções
def g1(x): return 1
def g2(x): return x
def g3(x): return x**2
def g4(x): return np.cos(x/5)

funcoes = [g1, g2, g3, g4]
phi, alfas = mmq(funcoes, dias, win)

In [ ]:
# Plotando a função phi
x = np.linspace(min(dias), 1, 200)
y = [phi(xi) for xi in x]

plt.figure(figsize=(12, 5))
plt.scatter(dias, win, label='Dados Reais', color='blue')
plt.xticks(dias)
plt.plot(x, y, label='Ajuste MMQ', color='skyblue', linestyle='--')
plt.legend()
plt.title("Ajuste via MMQ")
plt.show()

In [ ]:
x_pred = 1
print(f'Previsão para o dia 07/06 (x = {x_pred}): {phi(x_pred):.3f}')

### Erro da aproximação

O erro da aproximação é medido pela soma dos quadrados das diferenças entre os valores reais e os valores previstos pelo modelo ajustado:

$$
E = \sum_{i=1}^{n} \left( y_i - \phi(x_i) \right)^2
$$

Esse valor indica o quão bem a função ajustada representa os dados. Quanto menor o erro, melhor o ajuste.

### Análise dos resíduos

Os resíduos são definidos como a diferença entre os valores observados e os valores previstos pelo modelo:

$$
\text{Resíduo} = y_i - \phi(x_i)
$$

Se o gráfico dos resíduos mostrar curvas, agrupamentos ou tendências, isso pode indicar que o modelo não está capturando bem o comportamento dos dados.


In [ ]:
erro = sum((win[i] - phi(dias[i]))**2 for i in range(len(win)))
print(f'Erro quadrático total (EQ): {erro:.3f}')

eqm = erro / len(dias)
print(f"Erro quadrático médio (EQM): {eqm:.3f}")

In [ ]:
residuos = np.array([win[i] - phi(dias[i]) for i in range(len(dias))])

plt.figure(figsize=(12, 4))
plt.bar(dias, residuos)
plt.axhline(0, color='black', linestyle='--')
plt.title("Gráfico dos Resíduos")
plt.xlabel("Dia")
plt.ylabel("Resíduo")
plt.show()

### Intervalo de confiança da previsão

Quando estimamos a função ajustada $\phi(x)$ por mínimos quadrados, queremos mais do que apenas uma previsão pontual: também queremos saber quão confiável é essa previsão em cada ponto $x_0$.

Para isso, calculamos a **variância da previsão** $\text{Var}[\phi(x_0)]$ — ela quantifica a incerteza do modelo no ponto $x_0$, levando em conta tanto o ruído dos dados quanto a incerteza nos coeficientes $\alpha$ (que foram estimados com base nos dados).

### Cálculo da variância de previsão

O cálculo que usamos no código segue a seguinte expressão:

$$
\text{Var}[\phi(x_0)] = \bar{\sigma}^2 \cdot \boldsymbol{g}(x_0)^T A^{-1} \boldsymbol{g}(x_0)
$$

Onde:

- $\boldsymbol{g}(x_0)$ é o vetor coluna com as funções base avaliadas no ponto $x_0$:  
  $$ \boldsymbol{g}(x_0) = \begin{bmatrix} g_1(x_0) \\ g_2(x_0) \\ \vdots \\ g_p(x_0) \end{bmatrix} $$
  
- $A$ é a matriz usada no sistema normal:  
  $$ A = \sum_{i=1}^n \boldsymbol{g}(x_i) \boldsymbol{g}(x_i)^T $$
  
- $A^{-1}$ é a inversa da matriz $A$

- $\bar{\sigma}^2$ é a variância média dos resíduos, dada por:  
  $$ \bar{\sigma}^2 = \frac{1}{n - p} \text{Erro Quadrático} $$  
  Isso representa uma estimativa do ruído nos dados com base no erro quadrático total do ajuste.

A partir dessa variância, podemos construir o intervalo de confiança da previsão usando a normalidade dos resíduos:

$$
\phi(x_0) \pm z \cdot \sqrt{ \text{Var}[\phi(x_0)] }
$$

Onde $z$ depende do nível de confiança desejado (ex: $z \approx 1.96$ para 95%).

Esse intervalo mostra a faixa onde esperamos que a verdadeira média da resposta esteja, dado o modelo e os dados observados.

In [ ]:
# Cálculo da variância estimada
n = len(dias)
p = len(funcoes)
sigma2 = erro / (n - p)  # Variância estimada do erro

print("Variância estimada:", sigma2)

In [ ]:
# Inversa de A
A_inv = np.linalg.inv(A)

# Vetor g(x_pred)
g_pred = np.array([f(x_pred) for f in funcoes]).reshape(-1, 1)

# Variância da previsão
var_pred = sigma2 * (g_pred.T @ A_inv @ g_pred)[0, 0]
desvio_pred = np.sqrt(var_pred)
print("Desvio padrão previsto:", desvio_pred)

In [ ]:
# Intervalo para 95%

z = 1.96
lim_inf = phi(x_pred) - z * desvio_pred
lim_sup = phi(x_pred) + z * desvio_pred

print(f"Intervalo de 95% para x = {x_pred}: ({lim_inf:.3f}, {lim_sup:.3f})")

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(dias, win, color='blue', label='Dados Reais')
plt.plot(x, y, color='skyblue', linestyle='--', label='Ajuste Simples')
plt.axvline(x=1, color='gray', linestyle='--', alpha=0.7)
plt.errorbar(1, phi(x_pred), yerr=z * desvio_pred, fmt='o', color='black', label='Previsão x=1 com IC 95%')
plt.legend()
plt.title("Previsão com Intervalo de Confiança")
plt.show()

## Discussão e Considerações Finais

Embora o ajuste de curvas pelo método dos mínimos quadrados (MMQ) seja uma ferramenta poderosa, seu uso para prever séries temporais financeiras como esta possui limitações importantes.

- Os dados apresentam ruído significativo e oscilações não periódicas, o que dificulta a modelagem com funções simples.
- O comportamento do índice WIN não segue um padrão evidente, o que **prejudica modelos determinísticos** como os utilizados aqui.
- Seria mais apropriado empregar modelos **mais robustos e especializados**, como:
  - Modelos estatísticos de séries temporais (ex: ARIMA, SARIMA),
  - Métodos baseados em aprendizado de máquina (ex: regressão por árvores, redes neurais),
  - Ou modelos híbridos que considerem também variáveis externas e tendências de mercado.

Apesar disso, a experiência foi válida para compreender os fundamentos do MMQ, suas aplicações práticas e limitações em contextos reais.